In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# carregamento das imagens e caracterização

I1 = cv2.imread("imagem1.jpg", cv2.IMREAD_GRAYSCALE)
I2 = cv2.imread("imagem2.jpg", cv2.IMREAD_GRAYSCALE)

fig, ax = plt.subplots(1, 2, figsize=(12, 7))
ax[0].imshow(I1, cmap="gray", vmin=0, vmax=255); ax[0].set_title("I1")
ax[1].imshow(I2, cmap="gray", vmin=0, vmax=255); ax[1].set_title("I2")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

for nome, I in [("I1", I1), ("I2", I2)]:
    print(f"{nome}: largura={I.shape[1]}, altura={I.shape[0]}, tipo_dado={I.dtype}, "
          f"menor_intensidade={I.min()}, maior_intensidade={I.max()}, intensidade_média={I.mean():.2f}")

print("Mesmas dimensões?", I1.shape == I2.shape)

In [ ]:
# subtração
ID = I1.astype(np.int16) - I2.astype(np.int16)

print(f"Menor valor de ID: {ID.min()}")
print(f"Maior valor de ID: {ID.max()}")
print(f"Pixels negativos: {np.count_nonzero(ID < 0)}")
print(f"Pixels positivos: {np.count_nonzero(ID > 0)}")
print(f"Pixels zero: {np.count_nonzero(ID == 0)}")

erro_uint8 = I1 - I2

lim = np.abs(ID).max()
fig, ax = plt.subplots(1, 2, figsize=(16, 8))
im = ax[0].imshow(ID, cmap="seismic", vmin=-lim, vmax=lim)
ax[0].set_title("ID = I1 - I2 (int16)"); ax[0].axis("off")
plt.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].imshow(erro_uint8, cmap="gray"); ax[1].set_title("I1 - I2 em uint8 (errado)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# diferença absoluta (item 5)

IA = np.abs(ID)
IA_norm = cv2.normalize(IA.astype(np.float32), None, 0, 255,
                        cv2.NORM_MINMAX).astype(np.uint8)

fig, ax = plt.subplots(1, 4, figsize=(22, 7))
ax[0].imshow(I1, cmap="gray", vmin=0, vmax=255); ax[0].set_title("I1")
ax[1].imshow(I2, cmap="gray", vmin=0, vmax=255); ax[1].set_title("I2")
ax[2].imshow(ID, cmap="seismic", vmin=-lim, vmax=lim); ax[2].set_title("ID = I1 - I2")
ax[3].imshow(IA_norm, cmap="gray"); ax[3].set_title("IA = |I1 - I2| (normalizada)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# regiões modificadas
TD = 40
BD = np.where(IA > TD, 255, 0).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(14, 8))
ax[0].imshow(IA_norm, cmap="gray"); ax[0].set_title("Diferença absoluta IA")
ax[1].imshow(BD, cmap="gray", vmin=0, vmax=255); ax[1].set_title(f"Regiões modificadas (TD = {TD})")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print("TD =", TD)

In [ ]:
# combinação ponderada
alphas = [0.25, 0.50, 0.75]
fig, ax = plt.subplots(1, 3, figsize=(20, 8))
for a, alpha in zip(ax, alphas):
    IC = alpha * I1.astype(np.float32) + (1 - alpha) * I2.astype(np.float32)
    a.imshow(IC, cmap="gray", vmin=0, vmax=255)
    a.set_title(f"α = {alpha:.2f}"); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# mascaramento
h, w = I1.shape
M = np.zeros((h, w), dtype=np.uint8)
y0, y1 = int(0.00*h), int(0.28*h)
x0, x1 = int(0.38*w), int(0.66*w)
M[y0:y1, x0:x1] = 1

IM = I1 * M

fig, ax = plt.subplots(1, 3, figsize=(20, 8))
ax[0].imshow(I1, cmap="gray", vmin=0, vmax=255); ax[0].set_title("I1")
ax[1].imshow(M, cmap="gray", vmin=0, vmax=1);    ax[1].set_title("Máscara M")
ax[2].imshow(IM, cmap="gray", vmin=0, vmax=255); ax[2].set_title("IM = I1 · M")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# isolar o boneco usando BD como máscara
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
M2 = cv2.morphologyEx(BD, cv2.MORPH_CLOSE, kernel)
M2 = cv2.morphologyEx(M2, cv2.MORPH_OPEN, kernel)
M2 = (M2 > 0).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(20, 8))
ax[0].imshow(I2, cmap="gray", vmin=0, vmax=255); ax[0].set_title("I2")
ax[1].imshow(M2, cmap="gray");                   ax[1].set_title("Máscara derivada de BD")
ax[2].imshow(I2 * M2, cmap="gray", vmin=0, vmax=255); ax[2].set_title("I2 · M2 (objeto isolado)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# comparação final
IC_05 = (0.5 * I1.astype(np.float32) + 0.5 * I2.astype(np.float32))

fig, ax = plt.subplots(2, 3, figsize=(21, 14))
itens = [(I1, "I1"), (I2, "I2"), (IS_sat, "Soma (saturada)"),
         (IA_norm, "Diferença absoluta"), (IC_05, "Ponderada (α = 0.5)"),
         (IM, "Mascaramento")]
for a, (img, t) in zip(ax.ravel(), itens):
    a.imshow(img, cmap="gray", vmin=0, vmax=255); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()